In [11]:
# Install libraries (selaras requirements.txt dashboard)
%pip install pandas openpyxl scikit-learn transformers matplotlib seaborn torch -q


     ------------------------------------ 596.4/596.4 kB 412.2 kB/s eta 0:00:00
     -------------------------------------- 11.6/11.6 MB 476.5 kB/s eta 0:00:00
     -------------------------------------- 676.7/676.7 kB 1.1 MB/s eta 0:00:00
     -------------------------------------- 771.9/771.9 kB 1.7 MB/s eta 0:00:00
     ------------------------------------ 122.0/122.0 MB 723.1 kB/s eta 0:00:00
     -------------------------------------- 278.0/278.0 kB 2.9 MB/s eta 0:00:00
     -------------------------------------- 158.6/158.6 kB 2.4 MB/s eta 0:00:00
     -------------------------------------- 355.5/355.5 kB 1.3 MB/s eta 0:00:00
     ---------------------------------------- 2.7/2.7 MB 1.3 MB/s eta 0:00:00
     -------------------------------------- 122.7/122.7 kB 1.8 MB/s eta 0:00:00
     -------------------------------------- 119.2/119.2 kB 1.8 MB/s eta 0:00:00
     ------------------------------------ 203.9/203.9 kB 824.3 kB/s eta 0:00:00
     -------------------------------------


[notice] A new release of pip available: 22.2.2 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


# 1. BUSINESS UNDERSTANDING

### Latar Belakang
Subdirektorat Monitoring dan Evaluasi (MONEV) menerima ribuan tiket pertanyaan/keluhan pelanggan melalui live chat, email, dan media sosial setiap bulannya. Saat ini, belum ada analisis sentimen yang terstruktur untuk mengukur kepuasan pelanggan secara objektif. Kategori tiket yang diinput secara manual oleh agen sering kali tidak konsisten dan tidak mencerminkan sentimen riil pelanggan.

### Tujuan Bisnis
1. Mengukur tingkat kepuasan pelanggan berdasarkan analisis sentimen otomatis.
2. Mengidentifikasi kategori layanan dengan tingkat sentimen negatif tertinggi untuk prioritas perbaikan.
3. Menemukan kata kunci utama keluhan pelanggan (root cause).
4. Memberikan rekomendasi perbaikan layanan yang konkret dan terukur.

### Pertanyaan Bisnis
- Bagaimana distribusi sentimen (positif, netral, negatif) pelanggan di seluruh channel layanan?
- Kategori layanan apa yang memiliki persentase sentimen negatif tertinggi?
- Apa kata kunci yang paling sering muncul pada keluhan bersentimen negatif?

# 2. DATA UNDERSTANDING

Pada tahap ini, kita akan membaca semua file data dari berbagai channel (Live Chat, Email, dan Media Sosial), menggabungkannya, menstandardisasi kolom-kolom utama, dan menganalisis karakteristik awal data.

In [12]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

def try_read_csv(file_path, separators=[';', ','], encodings=['utf-8', 'latin1', 'cp1252']):
    for sep in separators:
        for encoding in encodings:
            try:
                df = pd.read_csv(file_path, sep=sep, encoding=encoding)
                print(f"   ✅ Berhasil membaca {file_path}: sep='{sep}', encoding='{encoding}'")
                return df
            except Exception:
                continue
    return None

file_configs = {
    'Live Chat': 'Report Ticket Multichat Monev_2026-07-13_08.43.17.csv',
    'Social Media': 'Report Ticket Sosmed Monev_2026-07-13_10.39.29.csv',
    'Email': 'Report Ticket Email Monev_2026-07-13_10.38.40.csv'
}

combined_dfs = []
for channel, file_name in file_configs.items():
    if os.path.exists(file_name):
        df_temp = try_read_csv(file_name)
        if df_temp is not None:
            df_temp['channel_source'] = channel
            combined_dfs.append(df_temp)
    else:
        print(f"⚠️ File {file_name} tidak ditemukan!")

df_raw = pd.concat(combined_dfs, ignore_index=True) if combined_dfs else pd.DataFrame()
print(f"\nTotal data mentah tergabung: {len(df_raw)} baris, {len(df_raw.columns)} kolom")

# Standardisasi nama kolom penting (lowercase dan strip)
df_raw.columns = df_raw.columns.str.strip().str.lower()

# Cek kolom penting
required_cols = ['ticket number', 'customer id', 'pertanyaan', 'jawaban', 'category', 'sub category']
print("\nCek ketersediaan kolom utama:")
for col in required_cols:
    status = "Ada" if col in df_raw.columns else "TIDAK ADA"
    print(f"- {col}: {status}")

# Distribusi data per channel
if 'channel_source' in df_raw.columns:
    print("\nDistribusi data per channel:")
    print(df_raw['channel_source'].value_counts())


   ✅ Berhasil membaca Report Ticket Multichat Monev_2026-07-13_08.43.17.csv: sep=';', encoding='latin1'
   ✅ Berhasil membaca Report Ticket Sosmed Monev_2026-07-13_10.39.29.csv: sep=';', encoding='latin1'
   ✅ Berhasil membaca Report Ticket Email Monev_2026-07-13_10.38.40.csv: sep=';', encoding='latin1'

Total data mentah tergabung: 6616 baris, 27 kolom

Cek ketersediaan kolom utama:
- ticket number: Ada
- customer id: Ada
- pertanyaan: Ada
- jawaban: Ada
- category: Ada
- sub category: Ada

Distribusi data per channel:
channel_source
Live Chat       5303
Email           1026
Social Media     287
Name: count, dtype: int64


# 3. DATA PREPARATION

Pipeline diselaraskan dengan **dashboard produksi** (`modules/cleaner.py`, `config/settings.py`):
1. Filter auto-reply / pesan sistem.
2. Hash `customer_id` untuk privasi.
3. Cleaning teks via `clean_and_validate_question()` (sama dengan Streamlit).
4. Parsing tanggal WIB untuk analisis tren waktu.


In [ ]:
import re
import hashlib
import sys
import os

sys.path.insert(0, os.path.abspath('.'))

from modules.cleaner import clean_and_validate_question, parse_frt_to_seconds

COL_STANDARD = {
    'ticket number': 'Ticket Number',
    'customer id': 'Customer ID',
    'pertanyaan': 'Pertanyaan',
    'jawaban': 'Jawaban',
    'category': 'Category',
    'sub category': 'Sub Category',
    'site name': 'Site Name',
    'created date': 'Created Date',
    'closed date': 'Closed Date',
    'channel': 'Channel',
    'frt': 'FRT',
    'ticket status': 'Ticket Status',
    'priority': 'Priority',
}

def standardize_columns(df):
    df = df.copy()
    df.columns = df.columns.str.strip().str.lower()
    rename = {k: v for k, v in COL_STANDARD.items() if k in df.columns}
    return df.rename(columns=rename)

AUTO_REPLY_PATTERNS = [
    r"Selamat datang di Bravo Bea Cukai",
    r"Disclaimer: Segala chat disini merupakan jawaban",
    r"Mohon tunggu beberapa saat untuk terhubung",
    r"Silakan tinggalkan pertanyaan Anda",
    r"Email ini telah diterima secara otomatis",
    r"Auto reply", r"Out of office", r"Balasan otomatis",
]
AUTO_REPLY_REGEX = re.compile("|".join(AUTO_REPLY_PATTERNS), re.IGNORECASE)

def is_auto_reply(text):
    if pd.isna(text) or str(text).strip() == "":
        return True
    return bool(AUTO_REPLY_REGEX.search(str(text)))

def hash_customer(cid):
    if pd.isna(cid) or str(cid).strip() == "":
        return "CUST_ANONYMOUS"
    return "CUST_" + hashlib.sha256(str(cid).strip().lower().encode()).hexdigest()[:8]

df_prep = standardize_columns(df_raw)

if 'Pertanyaan' in df_prep.columns:
    df_prep['is_auto'] = df_prep['Pertanyaan'].apply(is_auto_reply)
    n_before = len(df_prep)
    df_prep = df_prep[~df_prep['is_auto']].copy()
    print(f"Auto-reply: {n_before} -> {len(df_prep)} baris (buang {n_before - len(df_prep)})")

if 'Customer ID' in df_prep.columns:
    df_prep['customer_id_hashed'] = df_prep['Customer ID'].apply(hash_customer)

if 'Pertanyaan' in df_prep.columns:
    df_prep['pertanyaan_clean'] = df_prep['Pertanyaan'].apply(clean_and_validate_question)
    n_before = len(df_prep)
    df_prep = df_prep[df_prep['pertanyaan_clean'].notna()].copy()
    print(f"Validasi teks (clean_and_validate_question): {n_before} -> {len(df_prep)} baris")

if 'Created Date' in df_prep.columns:
    parsed = pd.to_datetime(df_prep['Created Date'], format='%Y-%m-%d %H.%M.%S', errors='coerce')
    nan_mask = parsed.isna() & df_prep['Created Date'].notna()
    if nan_mask.any():
        parsed.loc[nan_mask] = pd.to_datetime(df_prep['Created Date'].loc[nan_mask], errors='coerce')
    df_prep['Created Date'] = parsed
    df_prep['created_date_only'] = df_prep['Created Date'].dt.date

if 'FRT' in df_prep.columns:
    df_prep['frt_seconds'] = df_prep['FRT'].apply(parse_frt_to_seconds)

df_prep['text_clean'] = df_prep['pertanyaan_clean']

print(f"\nData akhir siap modeling: {len(df_prep)} baris")
if 'created_date_only' in df_prep.columns:
    d = df_prep['created_date_only'].dropna()
    if len(d):
        print(f"Rentang tanggal: {d.min()} s/d {d.max()}")

os.makedirs('output', exist_ok=True)
df_prep.to_csv('output/data_sentimen_prepared.csv', index=False)
print("✅ output/data_sentimen_prepared.csv disimpan (schema selaras dashboard).")


# 4. MODELING

Menggunakan pre-trained model RoBERTa Bahasa Indonesia khusus sentimen `w11wo/indonesian-roberta-base-sentiment-classifier` untuk mengklasifikasikan pertanyaan menjadi sentimen Positif, Netral, atau Negatif.

In [ ]:
from transformers import pipeline
from modules.cleaner import reclassify_sentiment

print("⏳ Memuat model sentimen Indo-RoBERTa...")
sentiment_pipeline = pipeline(
    "sentiment-analysis",
    model="w11wo/indonesian-roberta-base-sentiment-classifier"
)

df_model = pd.read_csv('output/data_sentimen_prepared.csv')
text_col = 'pertanyaan_clean' if 'pertanyaan_clean' in df_model.columns else 'text_clean'

def predict_sentiment_batch(texts, batch_size=32):
    labels, scores = [], []
    trimmed = [str(t)[:512] for t in texts]
    errors = 0
    for i in range(0, len(trimmed), batch_size):
        batch = trimmed[i:i + batch_size]
        try:
            preds = sentiment_pipeline(batch)
            labels.extend([p['label'].lower() for p in preds])
            scores.extend([float(p['score']) for p in preds])
        except Exception as e:
            print(f"⚠️ Error batch {i}: {e}")
            labels.extend(['neutral'] * len(batch))
            scores.extend([0.0] * len(batch))
            errors += len(batch)
    if errors:
        print(f"⚠️ Total baris fallback neutral karena error: {errors}")
    return labels, scores

print(f"⏳ Menganalisis sentimen untuk {len(df_model)} baris...")
raw_labels, raw_scores = predict_sentiment_batch(df_model[text_col].tolist())
df_model['sentiment_original'] = raw_labels
df_model['sentiment_score'] = raw_scores

df_model['sentiment'] = df_model.apply(
    lambda r: reclassify_sentiment(r[text_col], r['sentiment_original'], r['sentiment_score']),
    axis=1,
)

reclassified = (df_model['sentiment'] != df_model['sentiment_original']).sum()
print(f"✅ Selesai. Reklasifikasi heuristik: {reclassified} baris ({reclassified/len(df_model)*100:.1f}%)")
print(df_model['sentiment'].value_counts())
print("\nPerbandingan sebelum vs sesudah reklasifikasi:")
print(pd.crosstab(df_model['sentiment_original'], df_model['sentiment'], margins=True))

df_model.to_csv('output/sentiment_results.csv', index=False)
print("✅ output/sentiment_results.csv disimpan.")


# 5. EVALUATION

Evaluasi mencakup:
1. **Validasi keandalan model** (confidence score, prediksi rendah keyakinan, reklasifikasi)
2. **Insight bisnis** per channel, site, dan waktu
3. **Kata kunci keluhan actionable** (domain Bea Cukai + TF-IDF)
4. **Visualisasi dashboard** statis


In [ ]:
from collections import Counter
from sklearn.feature_extraction.text import TfidfVectorizer
from config.settings import (
    TECHNICAL_STATUS_PATTERNS,
    EXPLANATION_KEYWORDS,
    EMOTIONAL_NEGATIVE_PATTERNS,
)

df_model = pd.read_csv('output/sentiment_results.csv')
text_col = 'pertanyaan_clean' if 'pertanyaan_clean' in df_model.columns else 'text_clean'
df_model['sentiment'] = df_model['sentiment'].str.lower()

cat_col = 'Category' if 'Category' in df_model.columns else ('category' if 'category' in df_model.columns else None)
site_col = 'Site Name' if 'Site Name' in df_model.columns else ('site name' if 'site name' in df_model.columns else None)
channel_col = 'channel_source' if 'channel_source' in df_model.columns else ('Channel' if 'Channel' in df_model.columns else None)

total = len(df_model)
counts = df_model['sentiment'].value_counts()
print("=== DISTRIBUSI SENTIMEN (setelah reklasifikasi produksi) ===")
for label, val in counts.items():
    print(f"{label.upper():8}: {val:4} ({val/total*100:.1f}%)")

print("\n=== VALIDASI KEANDALAN MODEL ===")
if 'sentiment_score' in df_model.columns:
    sc = pd.to_numeric(df_model['sentiment_score'], errors='coerce')
    print(f"Rata-rata confidence : {sc.mean():.3f}")
    print(f"Median confidence    : {sc.median():.3f}")
    for thr in [0.5, 0.7, 0.9]:
        low = (sc < thr).sum()
        print(f"Prediksi confidence < {thr:.0%}: {low} ({low/len(df_model)*100:.1f}%)")

if 'sentiment_original' in df_model.columns:
    changed = df_model['sentiment'] != df_model['sentiment_original']
    print(f"\nBaris direklasifikasi heuristik: {changed.sum()} ({changed.mean()*100:.1f}%)")
    if changed.any():
        print("Contoh perubahan label (model -> final):")
        cols = [text_col, 'sentiment_original', 'sentiment', 'sentiment_score']
        for _, row in df_model.loc[changed, cols].head(5).iterrows():
            snippet = str(row[text_col])[:120].replace('\n', ' ')
            print(f"  [{row['sentiment_original']} -> {row['sentiment']}] (skor {row['sentiment_score']:.2f}) {snippet}...")

VALIDATION_SAMPLE_N = 150
export_cols = [c for c in ['Ticket Number', text_col, 'sentiment', 'sentiment_original', 'sentiment_score', cat_col, channel_col] if c and c in df_model.columns]
if len(df_model) > VALIDATION_SAMPLE_N:
    val_sample = df_model.sample(n=VALIDATION_SAMPLE_N, random_state=42)[export_cols].copy()
    val_sample['label_manual'] = ''
    val_sample['catatan_reviewer'] = ''
    val_sample.to_csv('output/validation_sample_manual.csv', index=False)
    print(f"\n✅ Sample validasi manual: output/validation_sample_manual.csv ({VALIDATION_SAMPLE_N} baris)")

print("\n=== SENTIMEN PER CHANNEL ===")
if channel_col:
    ch = pd.crosstab(df_model[channel_col], df_model['sentiment'], normalize='index') * 100
    ch_counts = df_model[channel_col].value_counts()
    for ch_name in ch.index:
        neg_pct = ch.loc[ch_name].get('negative', 0)
        pos_pct = ch.loc[ch_name].get('positive', 0)
        print(f"  {str(ch_name):15} | Vol: {ch_counts[ch_name]:5} | Neg: {neg_pct:5.1f}% | Pos: {pos_pct:5.1f}%")

site_grp = pd.DataFrame()
if site_col:
    print("\n=== TOP 10 SITE — PERSENTASE NEGATIF (min. 20 tiket) ===")
    site_grp = df_model.groupby(site_col).agg(
        total=('sentiment', 'count'),
        neg_pct=('sentiment', lambda s: (s == 'negative').mean() * 100),
    ).query('total >= 20').sort_values('neg_pct', ascending=False).head(10)
    for site, row in site_grp.iterrows():
        print(f"  {str(site)[:35]:35} | Tiket: {int(row['total']):4} | Neg: {row['neg_pct']:.1f}%")

daily = pd.DataFrame()
if 'created_date_only' in df_model.columns:
    df_model['created_date_only'] = pd.to_datetime(df_model['created_date_only'], errors='coerce')
    daily = df_model.groupby('created_date_only').agg(
        volume=('sentiment', 'count'),
        neg_pct=('sentiment', lambda s: (s == 'negative').mean() * 100),
    ).dropna()
    if len(daily):
        peak_neg_day = daily['neg_pct'].idxmax()
        print(f"\n=== TREN WAKTU ===")
        print(f"  Hari % negatif tertinggi: {peak_neg_day.date()} ({daily.loc[peak_neg_day, 'neg_pct']:.1f}%)")
        print(f"  Rata-rata % negatif harian: {daily['neg_pct'].mean():.1f}%")

if cat_col:
    print(f"\n=== TOP 10 KATEGORI — PERSENTASE NEGATIF (min. 5 tiket) ===")
    ct = pd.crosstab(df_model[cat_col], df_model['sentiment'], normalize='index') * 100
    cat_counts = df_model[cat_col].value_counts()
    valid_cats = cat_counts[cat_counts >= 5].index
    ct_f = ct.loc[valid_cats]
    if 'negative' in ct_f.columns:
        for idx, (cat, row) in enumerate(ct_f.sort_values('negative', ascending=False).head(10).iterrows(), 1):
            print(f" {idx:2}. {str(cat)[:40]:40} -> Neg: {row['negative']:.1f}% (n={cat_counts[cat]})")

print("\n=== KATA KUNCI KELUHAN ACTIONABLE ===")
neg_texts = df_model.loc[df_model['sentiment'] == 'negative', text_col].dropna().astype(str).tolist()

domain_hits = {}
for pat in TECHNICAL_STATUS_PATTERNS + EMOTIONAL_NEGATIVE_PATTERNS + EXPLANATION_KEYWORDS['negatif']:
    c = sum(1 for t in neg_texts if pat in t.lower())
    if c > 0:
        domain_hits[pat] = c
print("\n[Pola domain / emosi negatif]")
for pat, c in sorted(domain_hits.items(), key=lambda x: -x[1])[:15]:
    print(f"  {pat:25} : {c:4}x")

generic_stop = {
    'yang', 'dan', 'di', 'ke', 'dari', 'pada', 'akan', 'untuk', 'dengan', 'saya', 'kami',
    'ini', 'itu', 'atau', 'saja', 'sudah', 'belum', 'ada', 'tidak', 'bisa', 'pak', 'bu',
    'min', 'kak', 'admin', 'mohon', 'apakah', 'bagaimana', 'terima', 'kasih', 'bravo',
    'bea', 'cukai', 'nomor', 'tanggal', 'barang', 'dokumen', 'status', 'info', 'informasi',
}
if len(neg_texts) >= 10:
    tfidf = TfidfVectorizer(max_features=30, ngram_range=(1, 2), stop_words=list(generic_stop), min_df=3)
    X = tfidf.fit_transform(neg_texts)
    scores = X.sum(axis=0).A1
    terms = tfidf.get_feature_names_out()
    print("\n[TF-IDF top terms — keluhan negatif]")
    for term, score in sorted(zip(terms, scores), key=lambda x: -x[1])[:15]:
        print(f"  {term:30} : {score:.2f}")

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
colors = {'positive': '#2ecc71', 'neutral': '#95a5a6', 'negative': '#e74c3c'}
counts.plot(kind='pie', ax=axes[0, 0], autopct='%1.1f%%',
            colors=[colors.get(s, '#95a5a6') for s in counts.index], startangle=90)
axes[0, 0].set_title('Distribusi Sentimen', fontweight='bold')
axes[0, 0].set_ylabel('')

if channel_col:
    ch_neg = pd.crosstab(df_model[channel_col], df_model['sentiment'], normalize='index')['negative'] * 100
    ch_neg.sort_values().plot(kind='barh', ax=axes[0, 1], color='#e74c3c')
    axes[0, 1].set_title('% Negatif per Channel', fontweight='bold')
    axes[0, 1].set_xlabel('% Negatif')

if len(daily):
    ax = axes[1, 0]
    ax.bar(daily.index, daily['volume'], color='#3498db', alpha=0.7)
    ax2 = ax.twinx()
    ax2.plot(daily.index, daily['neg_pct'], color='#e74c3c', marker='o', linewidth=2)
    ax.set_title('Volume & % Negatif Harian', fontweight='bold')
    ax.set_ylabel('Volume')
    ax2.set_ylabel('% Negatif')
    ax.tick_params(axis='x', rotation=45)

if len(site_grp):
    site_grp['neg_pct'].sort_values().plot(kind='barh', ax=axes[1, 1], color='#9b59b6')
    axes[1, 1].set_title('Top Site — % Negatif', fontweight='bold')
elif cat_col and 'negative' in ct_f.columns:
    ct_f.sort_values('negative', ascending=True).tail(8)['negative'].plot(kind='barh', ax=axes[1, 1], color='#e74c3c')
    axes[1, 1].set_title('Top Kategori — % Negatif', fontweight='bold')

plt.tight_layout()
plt.savefig('output/sentiment_dashboard.png', dpi=300, bbox_inches='tight')
plt.show()
print("\n✅ Dashboard: output/sentiment_dashboard.png")


### 5.1 Validasi Manual (opsional)

Isi kolom `label_manual` pada `output/validation_sample_manual.csv` (positive / negative / neutral), lalu jalankan cell berikut untuk confusion matrix dan akurasi.


In [ ]:
from sklearn.metrics import classification_report, confusion_matrix
import seaborn as sns

val_path = 'output/validation_sample_manual.csv'
if os.path.exists(val_path):
    val_df = pd.read_csv(val_path)
    val_df['label_manual'] = val_df['label_manual'].astype(str).str.strip().str.lower()
    labeled = val_df[val_df['label_manual'].isin(['positive', 'negative', 'neutral'])]
    if len(labeled) >= 10:
        y_true = labeled['label_manual']
        y_pred = labeled['sentiment'].str.lower()
        print('=== LAPORAN KLASIFIKASI (manual vs model+heuristik) ===')
        print(classification_report(y_true, y_pred, digits=3))
        cm = confusion_matrix(y_true, y_pred, labels=['negative', 'neutral', 'positive'])
        fig, ax = plt.subplots(figsize=(6, 5))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                    xticklabels=['neg', 'neu', 'pos'], yticklabels=['neg', 'neu', 'pos'], ax=ax)
        ax.set_xlabel('Prediksi'); ax.set_ylabel('Label Manual')
        ax.set_title('Confusion Matrix — Validasi Manual')
        plt.tight_layout()
        plt.savefig('output/validation_confusion_matrix.png', dpi=200)
        plt.show()
        print(f'Akurasi: {(y_true == y_pred).mean()*100:.1f}%')
    else:
        print(f'Baris terlabel manual: {len(labeled)}. Isi minimal 10 baris.')
else:
    print('Jalankan cell evaluasi terlebih dahulu.')


# 6. DEPLOYMENT

### Integrasi Dashboard Produksi
- Hasil `output/sentiment_results.csv` kompatibel dengan **Streamlit dashboard** (`streamlit run app.py`).
- Preprocessing & reklasifikasi memakai modul **`modules/cleaner.py`** yang sama.

### Rekomendasi Aksi Bisnis
1. **Prioritas perbaikan**: kategori & site dengan % negatif tertinggi.
2. **Kata kunci actionable**: pola domain (`reject`, `pending`, `stuck`) untuk FAQ/SOP.
3. **Validasi berkala**: review `validation_sample_manual.csv` tiap periode laporan.
4. **Monitoring mingguan**: ekspor dashboard PNG + CSV ke manajemen MONEV.
